# Results

Thin wrapper over `emotion report`: it only reads `results/` and calls the report functions. Nothing is trained and the dataset is not loaded.
Regenerates `results/summary.csv`, `results/summary.md` and every figure in `figures/`.

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

from emotion_classifier.analysis import confusion_pairs, sample_errors
from emotion_classifier.data import LABELS
from emotion_classifier.plots import make_all, pooled_confusion, resolve_selection
from emotion_classifier.report import latest_runs, summarise, to_markdown, write_summary
from emotion_classifier.results import load_results

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS, FIGURES = ROOT / "results", ROOT / "figures"

# Which model each figure shows. Empty = best on validation macro-F1 (never test).
# Override with e.g. { "Transformer": "roberta_base" }.
OVERRIDES = {}

## Models shown in the figures

Chosen per family on **validation** macro-F1, so the test split is never used for selection.

In [ ]:
results = load_results( RESULTS )
selected = resolve_selection( results, OVERRIDES )
selected

## Summary table (test split)

Mean ± std over seeds for the transformers; the classical, baseline and zero-shot runs are single-seed.

In [ ]:
rows = summarise( results )
write_summary( rows, RESULTS / "summary.csv", RESULTS / "summary.md" )
display( Markdown( to_markdown( rows ) ) )

## Figures

Row-normalised confusion matrices (seeds pooled), per-class F1, and macro-F1 against inference cost.

In [ ]:
for path in make_all( results, FIGURES, selected ):
    display( Image( filename = str( path ) ) )

## Where each model goes wrong

The three most common confusions per model (counts pooled over seeds), with up to three example tweets each.
Examples are sampled reproducibly from the predictions of the model's first seed, so a pair can show fewer examples than its pooled count.

In [ ]:
def first_test_run( model ):
    runs = [ run for run in latest_runs( results, "test" ) if run.model == model ]
    return sorted( runs, key = lambda run: run.seed )[ 0 ]

for family, model in selected.items():
    run = first_test_run( model )
    print( f"=== { family }: { model } (examples from seed { run.seed }) ===" )
    for true_id, pred_id, count in confusion_pairs( pooled_confusion( results, model ), k = 3 ):
        print( f"{ LABELS[ true_id ] } -> { LABELS[ pred_id ] }: { count }" )
        for text in sample_errors( RESULTS / run.predictions_path, true_id, pred_id, n = 3 ):
            print( "   ", text )